In [1]:
# Import the libraries used throughout this practical:
# pandas for the table, numpy for math and random numbers,
# matplotlib for plots, scipy for the correlation test.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Fix the random seed so everyone gets exactly the same dataset
np.random.seed(42)

# Generate 100 students with normally distributed features
n = 100
study_hours = np.random.normal(5, 2, n)
attendance = np.random.normal(85, 10, n)
previous_mark = np.random.normal(70, 15, n)

# final_score is a weighted mix of the features plus random noise
final_score = ((0.5 * study_hours) + (0.3 * attendance) + (0.2 * previous_mark)
               + np.random.normal(0, 5, n))

df = pd.DataFrame({
    'study_hours': study_hours,
    'attendance': attendance,
    'previous_mark': previous_mark,
    'final_score': final_score
})

# Deliberately "dirty" the data, as the assignment specifies
df.loc[0:5, 'study_hours'] = np.nan   # 6 missing values
df.loc[99, 'study_hours'] = 50        # outlier
df.loc[98, 'attendance'] = -10        # impossible value

df.head(8)

,study_hours,attendance,previous_mark,final_score
0,NaN,70.846293,75.366810,35.178989
1,NaN,80.793547,78.411768,39.481248
2,NaN,81.572855,86.245769,48.605167
3,NaN,76.977227,85.807031,47.329456
4,NaN,83.387143,49.334959,37.044473
5,NaN,89.040509,55.932624,40.751177
6,8.158426,103.861859,77.725529,57.171201
7,6.534869,86.745778,77.706789,41.874669


## Part A Inspect and Clean

In [2]:
# Part A1: inspect the raw data to find what is "dirty" about it

# Shape = (number of rows, number of columns)
print("Raw shape:", df.shape)

# Count missing values (NaN) in each column
print("\nMissing values per column:")
print(df.isna().sum())

# Summary statistics: look at min and max for anything that looks impossible
print("\nSummary of the raw data:")
print(df.describe().round(2))

# The three largest study_hours values: the outlier should stand out clearly
print("\nThree largest study_hours values:")
print(df['study_hours'].nlargest(3))

# The three smallest attendance values: attendance cannot be negative
print("\nThree smallest attendance values:")
print(df['attendance'].nsmallest(3))

Raw shape: (100, 4)

Missing values per column:
study_hours      6
attendance       0
previous_mark    0
final_score      0
dtype: int64

Summary of the raw data:
       study_hours  attendance  previous_mark  final_score
count        94.00      100.00         100.00       100.00
mean          5.22       84.27          70.97        42.69
std           5.01       13.48          16.26         6.09
min          -0.24      -10.00          21.38        29.16
25%           3.73       76.83          60.17        38.19
50%           4.81       85.84          71.47        42.76
75%           5.74       90.38          80.57        46.69
max          50.00      112.20         127.79        57.17

Three largest study_hours values:
99    50.000000
31     8.704556
6      8.158426
Name: study_hours, dtype: float64

Three smallest attendance values:
98   -10.000000
10    65.812288
42    68.925168
Name: attendance, dtype: float64


The raw data has three problems:
- 6 missing study_hours values (rows 0-5)
- An outlier of 50 hours in study_hours (row 99), when every other value is under 9
- An impossible attendance value of -10 (row 98)

In [3]:
# Part A1 (cleaning): remove the bad records found in the inspection step.
# Work on a copy so the original raw dataframe `df` stays untouched.
clean = df.copy()

# 1) Remove the 50-hour outlier (row 99).
#    Every other student studies under 9 hours, so 50 is almost certainly a typo.
clean = clean[~(clean['study_hours'] > 30)]

# 2) Remove the impossible attendance value of -10 (row 98).
#    Attendance cannot be negative.
clean = clean[~(clean['attendance'] < 0)]

# 3) Drop the rows with missing study_hours (rows 0-5)
#    so every statistic below uses real, recorded values only.
clean = clean.dropna().reset_index(drop=True)

# Confirm how many rows were removed
print("Rows before:", len(df), "| rows after:", len(clean))
print("Rows removed:", len(df) - len(clean))

# Confirm the outlier is gone: the max study_hours should now be around 8.7
print("\nstudy_hours after cleaning: min =", round(clean['study_hours'].min(), 2),
      "max =", round(clean['study_hours'].max(), 2))

Rows before: 100 | rows after: 92
Rows removed: 8

study_hours after cleaning: min = -0.24 max = 8.7


In [4]:
# Part A2: separate the features from the target.
# Features (X) are the inputs we use to predict; the target (y) is what we predict.
features = ['study_hours', 'attendance', 'previous_mark']
target = 'final_score'

X = clean[features]   # feature matrix
y = clean[target]     # target vector

# Shape = (number of students, number of features)
print("Feature matrix X shape:", X.shape)
print("Target vector y shape: ", y.shape)

# Data types: float64 means the column is numerical
print("\nColumn dtypes:")
print(clean.dtypes)

Feature matrix X shape: (92, 3)
Target vector y shape:  (92,)

Column dtypes:
study_hours      float64
attendance       float64
previous_mark    float64
final_score      float64
dtype: object


In [5]:
# Other values that are still outside a sensible range after cleaning
print("attendance > 100    :", (clean['attendance'] > 100).sum(), "rows")
print("previous_mark > 100 :", (clean['previous_mark'] > 100).sum(), "rows")
print("study_hours < 0     :", (clean['study_hours'] < 0).sum(), "row")

attendance > 100    : 7 rows
previous_mark > 100 : 5 rows
study_hours < 0     : 1 row
